# Transformers, TP 2 : l'auto-attention construite pas à pas

Point de départ : une simple moyenne des tokens précédents. Point d'arrivée : une tête d'attention qui apprend elle-même quels tokens doivent compter.

Une idée revient dans tout le TP : une ligne de la matrice de poids dit combien chaque token passé contribue au token courant, et la somme de cette ligne vaut toujours 1.

À rendre : les quatre versions, les tests torch.allclose, la matrice wei[0] et une explication des formes B, T, C et d_h.

In [1]:
import torch
import torch.nn as nn
from torch.nn import functional as F

## 1. Agréger avec un produit matriciel

Une matrice triangulaire inférieure dont chaque ligne est normalisée, multipliée par b, donne à la ligne t la moyenne des t premières lignes de b.

In [2]:
torch.manual_seed(42)

a = torch.tril(torch.ones(3, 3))
a = a / a.sum(dim=1, keepdim=True)
b = torch.randint(0, 10, (3, 2)).float()
c = a @ b

for nom, m in (("a", a), ("b", b), ("c = a @ b", c)):
    print(f"{nom} =\n{m}\n")

a =
tensor([[1.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000],
        [0.3333, 0.3333, 0.3333]])

b =
tensor([[2., 7.],
        [6., 4.],
        [6., 5.]])

c = a @ b =
tensor([[2.0000, 7.0000],
        [4.0000, 5.5000],
        [4.6667, 5.3333]])



### Questions

*Q1. Calculer c₂ à la main.*

Avec b = [[2, 7], [6, 4], [6, 5]], la 2ᵉ ligne de a vaut (½, ½, 0). Donc c₂ = ½·b₁ + ½·b₂ = ((2 + 6)/2, (7 + 4)/2) = (4 ; 5,5). Même calcul pour la 3ᵉ ligne : ((2 + 6 + 6)/3, (7 + 4 + 5)/3) ≈ (4,67 ; 5,33). On retrouve bien la sortie.

*Q2. Pourquoi chaque ligne de a somme-t-elle à 1 ?*

Parce qu'on divise chaque ligne de tril(ones) par sa propre somme. La ligne t contient t fois la valeur 1/t : on calcule une moyenne des vecteurs visibles, pas une somme qui grossirait avec t.

## 2. Moyenne du passé, version boucle

Version de référence, lente mais facile à lire : pour chaque séquence b et chaque position t, on moyenne les vecteurs des positions 0 à t.

In [3]:
torch.manual_seed(1337)
B, T, C = 4, 8, 2
x = torch.randn(B, T, C)

xbow = torch.zeros((B, T, C))
for b in range(B):
    for t in range(T):
        xprev = x[b, :t + 1]               # (t+1, C)
        xbow[b, t] = xprev.mean(dim=0)

for t in (0, 3, 7):
    print(f"t = {t} : xprev a la forme {tuple(x[0, :t + 1].shape)}")

t = 0 : xprev a la forme (1, 2)
t = 3 : xprev a la forme (4, 2)
t = 7 : xprev a la forme (8, 2)


### Questions

*Q1. Forme de xprev pour t = 0, 3 et 7 ?*

x[b, :t+1] a la forme (t+1, C) : (1, 2) pour t = 0, (4, 2) pour t = 3 et (8, 2) pour t = 7, comme l'affiche la cellule.

*Q2. Pourquoi moyenner sur dim=0 ?*

Une fois b fixé, l'axe 0 de xprev est celui des positions (le temps) et l'axe 1 celui des composantes. On veut moyenner les tokens pour obtenir un seul vecteur de taille C, celui qui est rangé dans xbow[b, t].

## 3. Vectoriser

Deux écritures sans boucle donnent exactement le même résultat. La première réutilise la matrice triangulaire normalisée. La seconde part de scores nuls, met −∞ sur le futur et passe le tout dans un softmax : c'est cette forme qu'utilisera l'attention.

### Version 2 : produit matriciel

In [4]:
wei = torch.tril(torch.ones(T, T))
wei = wei / wei.sum(dim=1, keepdim=True)
xbow2 = wei @ x
print("version 2 == boucle :", torch.allclose(xbow, xbow2, atol=1e-6))

version 2 == boucle : True


### Version 3 : masque puis softmax

In [5]:
tril = torch.tril(torch.ones(T, T))
wei = torch.zeros((T, T)).masked_fill(tril == 0, float("-inf"))
wei = F.softmax(wei, dim=-1)
xbow3 = wei @ x
print("version 3 == boucle :", torch.allclose(xbow, xbow3, atol=1e-6))
print(wei)

version 3 == boucle : True
tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3333, 0.3333, 0.3333, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2500, 0.2500, 0.2500, 0.2500, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.0000, 0.0000, 0.0000],
        [0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.0000, 0.0000],
        [0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.0000],
        [0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250]])


### Questions

*Q1. Pourquoi wei @ x marche-t-il avec une matrice (T, T) et un tenseur (B, T, C) ?*

L'opérateur @ multiplie les deux dernières dimensions et considère celles d'avant comme des dimensions de batch. wei est donc répété implicitement sur les B séquences : (T, T) @ (B, T, C) se comporte comme (B, T, T) @ (B, T, C) et donne (B, T, C). Les mêmes poids s'appliquent à chaque séquence.

*Q2. Que devient le softmax aux positions à −∞ ?*

exp(−∞) = 0 : ces positions reçoivent un poids exactement nul. Les positions restantes ont toutes un score 0 et se partagent le poids à égalité, soit 1/(t+1) chacune. On retrouve la moyenne causale, le futur est invisible et chaque ligne somme toujours à 1.

## 4. Une vraie tête d'attention

Les poids ne sont plus fixés : ils viennent du produit entre les requêtes q et les clés k, mis à l'échelle par 1/√d_h, masqués puis normalisés. La sortie mélange les valeurs v.

In [6]:
torch.manual_seed(1337)
B, T, C = 4, 8, 32
head_size = 16
x = torch.randn(B, T, C)

key = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)

k = key(x)                                         # (B, T, d_h)
q = query(x)                                       # (B, T, d_h)
wei = q @ k.transpose(-2, -1) * head_size ** -0.5  # (B, T, T), mis à l'échelle
tril = torch.tril(torch.ones(T, T))
wei = wei.masked_fill(tril == 0, float("-inf"))
wei = F.softmax(wei, dim=-1)

v = value(x)                                       # (B, T, d_h)
out = wei @ v                                      # (B, T, d_h)

torch.set_printoptions(precision=3, sci_mode=False)
print("wei[0] =\n", wei[0])
print("\nsomme de chaque ligne :", wei[0].sum(dim=-1))
print("poids vers le futur nuls :", bool((wei[0].triu(diagonal=1) == 0).all()))
print("forme de out :", tuple(out.shape))

wei[0] =
 tensor([[1.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000],
        [0.397, 0.603, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000],
        [0.307, 0.289, 0.404, 0.000, 0.000, 0.000, 0.000, 0.000],
        [0.323, 0.218, 0.244, 0.215, 0.000, 0.000, 0.000, 0.000],
        [0.148, 0.203, 0.166, 0.146, 0.337, 0.000, 0.000, 0.000],
        [0.126, 0.249, 0.132, 0.106, 0.314, 0.072, 0.000, 0.000],
        [0.160, 0.199, 0.114, 0.112, 0.142, 0.167, 0.106, 0.000],
        [0.084, 0.120, 0.108, 0.154, 0.109, 0.115, 0.156, 0.155]],
       grad_fn=<SelectBackward0>)

somme de chaque ligne : tensor([1.000, 1.000, 1.000, 1.000, 1.000, 1.000, 1.000, 1.000],
       grad_fn=<SumBackward1>)
poids vers le futur nuls : True
forme de out : (4, 8, 16)


### Questions

*Q1. Quelle ligne de wei[0] correspond au dernier token ?*

La dernière, wei[0][-1] (indice T − 1 = 7). C'est la seule ligne sans zéro, puisque le dernier token voit les 8 positions.

*Q2. Pourquoi la sortie mélange-t-elle v et pas directement k ?*

Les trois projections n'ont pas le même rôle. Le produit q·k sert seulement à décider combien chaque token compte. v porte l'information qui est réellement transmise une fois ce choix fait. En séparant k et v, un token peut se faire « trouver » grâce à une propriété et transmettre une information différente.

## 5. Pourquoi diviser par √d_h

Si q et k ont des composantes de variance 1, leur produit scalaire a une variance d'environ d_h. Des scores aussi dispersés rendent le softmax presque « one-hot ». La division par √d_h ramène la variance vers 1.

In [7]:
torch.manual_seed(1337)
k = torch.randn(B, T, head_size)
q = torch.randn(B, T, head_size)

brut = q @ k.transpose(-2, -1)
normalise = brut * head_size ** -0.5
print(f"variance sans mise à l'échelle : {brut.var().item():.2f}")
print(f"variance avec mise à l'échelle : {normalise.var().item():.2f}")

z = torch.tensor([0.1, -0.2, 0.3, -0.2, 0.5])
print("\nsoftmax(z)     :", torch.softmax(z, dim=-1))
print("softmax(8 * z) :", torch.softmax(z * 8, dim=-1))

variance sans mise à l'échelle : 18.44
variance avec mise à l'échelle : 1.15

softmax(z)     : tensor([0.192, 0.143, 0.235, 0.143, 0.287])
softmax(8 * z) : tensor([0.033, 0.003, 0.162, 0.003, 0.800])


### Bilan du TP 2

*Vérifications.*

La cellule de l'étape 4 l'affiche directement : chaque ligne de wei[0] somme à 1, tout ce qui est au-dessus de la diagonale vaut 0 (aucun poids vers le futur), et out a la forme (4, 8, 16), c'est-à-dire (B, T, d_h).

*Rôle de Q, K et V.*

La requête q représente ce que le token courant cherche. La clé k représente ce que chaque token propose pour être trouvé ; le produit q·k mesure la compatibilité entre les deux. La valeur v est l'information transmise : la sortie est la moyenne des v pondérée par softmax(q·kᵀ/√d_h).

*Mise à l'échelle.*

Sans division, la variance des scores vaut 18,4, du même ordre que d_h = 16 ; avec la division, elle tombe à 1,15, proche de 1. L'exemple de z montre l'effet sur le softmax : multiplier les scores par 8 fait passer le plus grand poids de 0,29 à 0,80. Sans mise à l'échelle, un seul token écraserait les autres et les gradients vers ces autres tokens deviendraient presque nuls, ce qui ralentit l'apprentissage.